In [ ]:
# 3 - PageRank Implementation

In [1]:
!nvidia-smi

Mon Jun  1 08:39:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
from numba import cuda
import time
from google.colab import files

In [3]:
uploaded = files.upload()

Saving CollegeMsg.txt to CollegeMsg.txt


In [4]:
edges = []

with open("CollegeMsg.txt", "r") as f:

    for line in f:

        parts = line.strip().split()

        if len(parts) < 2:
            continue

        src = int(parts[0]) - 1
        dst = int(parts[1]) - 1

        edges.append((src, dst))

edges = np.array(edges, dtype=np.int32)

print("Total Edges:", len(edges))

Total Edges: 59835


In [5]:
num_nodes = int(edges.max()) + 1

print("Total Nodes:", num_nodes)

Total Nodes: 1899


In [6]:
out_degree = np.zeros(
    num_nodes,
    dtype=np.int32
)

for src, dst in edges:
    out_degree[src] += 1

print("Out Degree Computed")

Out Degree Computed


In [7]:
pagerank = np.ones(
    num_nodes,
    dtype=np.float32
) / num_nodes

damping = 0.85

iterations = 20

In [8]:
@cuda.jit
def pagerank_kernel(
        edges,
        out_degree,
        old_rank,
        new_rank):

    idx = cuda.grid(1)

    if idx < edges.shape[0]:

        src = edges[idx, 0]
        dst = edges[idx, 1]

        degree = out_degree[src]

        if degree > 0:

            contribution = (
                old_rank[src] /
                degree
            )

            cuda.atomic.add(
                new_rank,
                dst,
                contribution
            )

In [9]:
d_edges = cuda.to_device(edges)

d_out_degree = cuda.to_device(out_degree)

In [10]:
threads_per_block = 256

blocks_per_grid = (
    len(edges)
    + threads_per_block
    - 1
) // threads_per_block

start = time.time()

for i in range(iterations):

    new_rank = np.zeros(
        num_nodes,
        dtype=np.float32
    )

    d_old = cuda.to_device(
        pagerank
    )

    d_new = cuda.to_device(
        new_rank
    )

    pagerank_kernel[
        blocks_per_grid,
        threads_per_block
    ](
        d_edges,
        d_out_degree,
        d_old,
        d_new
    )

    cuda.synchronize()

    new_rank = d_new.copy_to_host()

    pagerank = (
        (1 - damping)
        / num_nodes
        + damping * new_rank
    )

end = time.time()

In [11]:
top_nodes = np.argsort(
    -pagerank
)[:10]

print("\nTop 10 Nodes\n")

for i, node in enumerate(top_nodes):

    print(
        f"Rank {i+1} "
        f"Node {node+1} "
        f"Score {pagerank[node]:.6f}"
    )


Top 10 Nodes

Rank 1 Node 32 Score 0.004741
Rank 2 Node 323 Score 0.004735
Rank 3 Node 372 Score 0.004213
Rank 4 Node 103 Score 0.003971
Rank 5 Node 1624 Score 0.003833
Rank 6 Node 325 Score 0.003445
Rank 7 Node 542 Score 0.003421
Rank 8 Node 42 Score 0.003411
Rank 9 Node 72 Score 0.003280
Rank 10 Node 454 Score 0.003212


In [12]:
print(
    "\nExecution Time:",
    round(end-start,4),
    "seconds"
)


Execution Time: 4.4589 seconds
